## Exploratory Data Analysis for the Vancouver Non-Market Housing Dashboard

## Loading and Checking Data

In [1]:
import pandas as pd
import altair as alt
import geopandas as gpd
from shapely.geometry import Point, shape
import json

### Housing Data

In [2]:
housing = pd.read_csv(
    "../data/raw/non-market-housing.csv",
    sep=";",
    dtype={"Project Status": "category", "Occupancy Year": "Int64"}
)
housing

,Index Number,Name,Address,Project Status,Occupancy Year,Operator,Clientele- Families,Clientele - Seniors,Clientele - Other,Design - Accessible 1BR,...,Design - Adaptable 3BR,Design - Adaptable 4BR,Design - Standard 1BR,Design - Standard 2BR,Design - Standard 3BR,Design - Standard 4BR,Design - Standard Studio,Design - Standard Room,URL,Geom
0,6,Helen's Court Co-op,2137 W 1st Ave,Completed,1984,Helen's Court Co-op Housing Association,35,0,9,2.0,...,0.0,0.0,7.0,22.0,13.0,0.0,0.0,0.0,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.1536261, 49.27104183], ""..."
1,7,Laura Jamieson Co-op,1349 E 2nd Ave,Completed,1987,Laura Jamieson Co-op Housing Association,42,0,5,1.0,...,0.0,0.0,4.0,23.0,18.0,0.0,0.0,0.0,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.07614157, 49.26900321], ..."
2,8,Westerdale Co-op,1507 E 2nd Ave,Completed,1984,Westerdale Co-op Housing Association,10,0,9,4.0,...,0.0,0.0,5.0,6.0,2.0,0.0,0.0,0.0,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.07304966, 49.26896961], ..."
3,18,Vancouver Native (4th Ave),1560 E 4th Ave,Completed,1989,BC Indigenous Housing Society,20,10,1,1.0,...,0.0,0.0,10.0,11.0,4.0,5.0,0.0,0.0,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.0718679, 49.2666326], ""t..."
4,20,Northern Way Co-op,675 E 5th Ave,Completed,1985,Northern Way Co-op Housing Association,44,0,16,3.0,...,0.0,0.0,13.0,24.0,19.0,1.0,0.0,0.0,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.09065341, 49.26639799], ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
636,929,NaN,998 E 19th Ave,Approved,<NA>,NaN,0,0,105,22.0,...,NaN,NaN,37.0,26.0,12.0,NaN,30.0,NaN,NaN,"{""coordinates"": [-123.08438908, 49.25336844], ..."
637,934,NaN,1710-1730 E Pender St,Approved,<NA>,Lu'Ma Native Housing Society,71,0,120,NaN,...,NaN,NaN,120.0,38.0,28.0,5.0,NaN,NaN,NaN,"{""coordinates"": [-123.07002235, 49.28001135], ..."
638,948,Brennan's Place,545 E Cordova,Completed,2024,Lookout Housing and Health Society,0,0,20,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,20.0,NaN,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.0923813, 49.28240033], ""..."
639,989,Murray Hotel,1119 Hornby,Completed,2017,Atira Women’s Resource Society,0,0,95,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,95.0,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.12750154, 49.27940628], ..."


In [3]:
housing = housing.rename(columns={"Clientele- Families": "Clientele - Families"})

#### Null Values

In [4]:
housing.isnull().sum()

Index Number                    0
Name                           12
Address                         0
Project Status                  0
Occupancy Year                 63
Operator                       27
Clientele - Families            0
Clientele - Seniors             0
Clientele - Other               0
Design - Accessible 1BR       126
Design - Accessible 2BR       140
Design - Accessible 3BR       153
Design - Accessible 4BR       159
Design - Accessible Studio    129
Design - Accessible Room      156
Design - Adaptable 1BR        155
Design - Adaptable 2BR        158
Design - Adaptable 3BR        159
Design - Adaptable 4BR        160
Design - Standard 1BR          63
Design - Standard 2BR          66
Design - Standard 3BR          77
Design - Standard 4BR         133
Design - Standard Studio       43
Design - Standard Room        107
URL                            50
Geom                           19
dtype: int64

From above, we see that there are null values in the dataset. This is expected in some cases; occupancy year will naturally be null if there are no current occupants in the building, and name, operator, and URL may not currently be available depending on the current stage of the project.

In [5]:
null_cols = [
    "Name",
    "Operator",
    "URL"
]

housing[(housing[null_cols].isnull().any(axis=1)) & (housing["Occupancy Year"].notnull())]

,Index Number,Name,Address,Project Status,Occupancy Year,Operator,Clientele - Families,Clientele - Seniors,Clientele - Other,Design - Accessible 1BR,...,Design - Adaptable 3BR,Design - Adaptable 4BR,Design - Standard 1BR,Design - Standard 2BR,Design - Standard 3BR,Design - Standard 4BR,Design - Standard Studio,Design - Standard Room,URL,Geom


For the most part, null values in the columns related to Design are likely equivalent to 0; we see below that with the exception of one project, all other projects have some design column that is not null. As the one project without any values in any of the design columns has a status of proposed, it is possible that information on the exact design is not currently available.

For this reason, with the exception of the one project, we impute null values in the design columns with zero.

In [6]:
design_cols = [col for col in housing.columns if col.startswith("Design")]
housing[housing[design_cols].isnull().all(axis=1)]

,Index Number,Name,Address,Project Status,Occupancy Year,Operator,Clientele - Families,Clientele - Seniors,Clientele - Other,Design - Accessible 1BR,...,Design - Adaptable 3BR,Design - Adaptable 4BR,Design - Standard 1BR,Design - Standard 2BR,Design - Standard 3BR,Design - Standard 4BR,Design - Standard Studio,Design - Standard Room,URL,Geom
635,926,Shawn Oaks,5505 Oak St,Proposed,<NA>,Affordable Housing Societies,63,0,117,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""coordinates"": [-123.12901006, 49.23571799], ..."


In [7]:
# impute missing values in design columns with 0
mask = ~housing[design_cols].isnull().all(axis=1)
housing[design_cols] = housing[design_cols].where(~mask, housing[design_cols].fillna(0))

This leaves the null values in Geom. There is no reason for Geom to be empty, since all projects have an address. It should be possible to obtain the coordinates from the address in Python using packages such as `geopy`, but for the time being, we will remove them.

#### Data Types

In [8]:
housing.dtypes

Index Number                     int64
Name                               str
Address                            str
Project Status                category
Occupancy Year                   Int64
Operator                           str
Clientele - Families             int64
Clientele - Seniors              int64
Clientele - Other                int64
Design - Accessible 1BR        float64
Design - Accessible 2BR        float64
Design - Accessible 3BR        float64
Design - Accessible 4BR        float64
Design - Accessible Studio     float64
Design - Accessible Room       float64
Design - Adaptable 1BR         float64
Design - Adaptable 2BR         float64
Design - Adaptable 3BR         float64
Design - Adaptable 4BR         float64
Design - Standard 1BR          float64
Design - Standard 2BR          float64
Design - Standard 3BR          float64
Design - Standard 4BR          float64
Design - Standard Studio       float64
Design - Standard Room         float64
URL                      

In [9]:
# convert design columns to integer types
housing[design_cols] = housing[design_cols].astype("Int64")

In [10]:
housing["Geom"] = housing["Geom"].apply(lambda x: shape(json.loads(x)) if pd.notnull(x) else None)

### Get Local Area

We obtain the local area of each project by joining the dataframe with data on the local area boundaries in Vancouver:

In [11]:
bounds = pd.read_csv(
    "../data/raw/local-area-boundary.csv",
    sep=";"
)

bound_geometries = bounds["Geom"].apply(lambda x: shape(json.loads(x)))

bounds = gpd.GeoDataFrame(
    bounds[["Name"]], 
    geometry=bound_geometries,
    crs="EPSG:4326"
)

bounds

,Name,geometry
0,Kensington-Cedar Cottage,"POLYGON ((-123.05659 49.26198, -123.05663 49.2..."
1,Kitsilano,"POLYGON ((-123.13768 49.27532, -123.14375 49.2..."
2,Riley Park,"POLYGON ((-123.10562 49.23312, -123.11617 49.2..."
3,Shaughnessy,"POLYGON ((-123.15527 49.23452, -123.15508 49.2..."
4,Victoria-Fraserview,"POLYGON ((-123.05683 49.2042, -123.05846 49.20..."
5,West Point Grey,"POLYGON ((-123.22445 49.27892, -123.20515 49.2..."
6,Arbutus Ridge,"POLYGON ((-123.1526 49.25723, -123.16488 49.25..."
7,Grandview-Woodland,"POLYGON ((-123.07702 49.29025, -123.06778 49.2..."
8,Killarney,"POLYGON ((-123.02356 49.20015, -123.03998 49.2..."
9,Strathcona,"POLYGON ((-123.09929 49.28927, -123.0939 49.29..."


In [12]:
point = housing["Geom"][0]
boundary = bounds["geometry"][0]

boundary.contains(point)

False

In [13]:
df = housing.copy()
df["Name"] = None

for idx, proj in housing.iterrows():
    point = proj["Geom"]
    
    for bound_idx in bounds.index:
        polygon = bounds.loc[bound_idx, "geometry"]
        if polygon.contains(point):
            df.at[idx, "Local Area"] = bounds.loc[bound_idx, "Name"]
            break

In [14]:
df["Local Area"] = df["Local Area"].fillna("Other")
df

,Index Number,Name,Address,Project Status,Occupancy Year,Operator,Clientele - Families,Clientele - Seniors,Clientele - Other,Design - Accessible 1BR,...,Design - Adaptable 4BR,Design - Standard 1BR,Design - Standard 2BR,Design - Standard 3BR,Design - Standard 4BR,Design - Standard Studio,Design - Standard Room,URL,Geom,Local Area
0,6,None,2137 W 1st Ave,Completed,1984,Helen's Court Co-op Housing Association,35,0,9,2,...,0,7,22,13,0,0,0,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.1536261 49.27104183),Kitsilano
1,7,None,1349 E 2nd Ave,Completed,1987,Laura Jamieson Co-op Housing Association,42,0,5,1,...,0,4,23,18,0,0,0,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.07614157 49.26900321),Grandview-Woodland
2,8,None,1507 E 2nd Ave,Completed,1984,Westerdale Co-op Housing Association,10,0,9,4,...,0,5,6,2,0,0,0,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.07304966 49.26896961),Grandview-Woodland
3,18,None,1560 E 4th Ave,Completed,1989,BC Indigenous Housing Society,20,10,1,1,...,0,10,11,4,5,0,0,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.0718679 49.2666326),Grandview-Woodland
4,20,None,675 E 5th Ave,Completed,1985,Northern Way Co-op Housing Association,44,0,16,3,...,0,13,24,19,1,0,0,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.09065341 49.26639799),Mount Pleasant
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
636,929,None,998 E 19th Ave,Approved,<NA>,NaN,0,0,105,22,...,0,37,26,12,0,30,0,NaN,POINT (-123.08438908 49.25336844),Kensington-Cedar Cottage
637,934,None,1710-1730 E Pender St,Approved,<NA>,Lu'Ma Native Housing Society,71,0,120,0,...,0,120,38,28,5,0,0,NaN,POINT (-123.07002235 49.28001135),Grandview-Woodland
638,948,None,545 E Cordova,Completed,2024,Lookout Housing and Health Society,0,0,20,0,...,0,0,0,0,0,20,0,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.0923813 49.28240033),Strathcona
639,989,None,1119 Hornby,Completed,2017,Atira Women’s Resource Society,0,0,95,0,...,0,0,0,0,0,0,95,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.12750154 49.27940628),Downtown


### Design Type Counts

As we are interested in the overall design type (Accessible, Adaptable, and Standard), we can create new columns to aggregate the counts for each project.

In [15]:
adap_cols = []
acc_cols = []
std_cols = []

for col in df.columns:
    if "Adaptable" in col:
        adap_cols.append(col)
    elif "Accessible" in col:
        acc_cols.append(col)
    elif "Standard" in col:
        std_cols.append(col)

df["Adaptable"] = df[adap_cols].sum(axis=1)
df["Accessible"] = df[acc_cols].sum(axis=1)
df["Standard"] = df[std_cols].sum(axis=1)

In [16]:
df

,Index Number,Name,Address,Project Status,Occupancy Year,Operator,Clientele - Families,Clientele - Seniors,Clientele - Other,Design - Accessible 1BR,...,Design - Standard 3BR,Design - Standard 4BR,Design - Standard Studio,Design - Standard Room,URL,Geom,Local Area,Adaptable,Accessible,Standard
0,6,None,2137 W 1st Ave,Completed,1984,Helen's Court Co-op Housing Association,35,0,9,2,...,13,0,0,0,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.1536261 49.27104183),Kitsilano,0,2,42
1,7,None,1349 E 2nd Ave,Completed,1987,Laura Jamieson Co-op Housing Association,42,0,5,1,...,18,0,0,0,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.07614157 49.26900321),Grandview-Woodland,0,2,45
2,8,None,1507 E 2nd Ave,Completed,1984,Westerdale Co-op Housing Association,10,0,9,4,...,2,0,0,0,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.07304966 49.26896961),Grandview-Woodland,0,6,13
3,18,None,1560 E 4th Ave,Completed,1989,BC Indigenous Housing Society,20,10,1,1,...,4,5,0,0,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.0718679 49.2666326),Grandview-Woodland,0,1,30
4,20,None,675 E 5th Ave,Completed,1985,Northern Way Co-op Housing Association,44,0,16,3,...,19,1,0,0,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.09065341 49.26639799),Mount Pleasant,0,3,57
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
636,929,None,998 E 19th Ave,Approved,<NA>,NaN,0,0,105,22,...,12,0,30,0,NaN,POINT (-123.08438908 49.25336844),Kensington-Cedar Cottage,0,25,105
637,934,None,1710-1730 E Pender St,Approved,<NA>,Lu'Ma Native Housing Society,71,0,120,0,...,28,5,0,0,NaN,POINT (-123.07002235 49.28001135),Grandview-Woodland,0,0,191
638,948,None,545 E Cordova,Completed,2024,Lookout Housing and Health Society,0,0,20,0,...,0,0,20,0,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.0923813 49.28240033),Strathcona,0,0,20
639,989,None,1119 Hornby,Completed,2017,Atira Women’s Resource Society,0,0,95,0,...,0,0,0,95,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.12750154 49.27940628),Downtown,0,0,95


We write the data to `data/processed` for later use.

In [17]:
df.to_csv("../data/processed/clean-non-market-housing.csv")

## Visualizations

We will be using the following user story from our proposal:

2. Demographic Equity Analysis
    - **User Story**: A policy analyst wants to filter by clientele type (families, seniors, other) so they can determine whether developments align with demographic needs.
    - **JTBD**: When reviewing housing policies, the policy analyst wants to compare family and senior unit counts across neighbourhoods so they can prioritize appropriate housing types.

In [18]:
# pivot data to easily access counts

proj_cols = [
    'Index Number',
    'Name',
    'Address',
    'Project Status',
    'Occupancy Year',
    'Operator',
    'URL',
    'Geom',
    'Local Area'
]

df = df.melt(id_vars=proj_cols, var_name="feature", value_name="unit_count")
df[["unit_category", "unit_detail"]] = df["feature"].str.split(" - ", n=1, expand=True)
df = df.drop(columns=["feature"])
df = df[df["unit_count"].notna() & (df["unit_count"] > 0)]

df

,Index Number,Name,Address,Project Status,Occupancy Year,Operator,URL,Geom,Local Area,unit_count,unit_category,unit_detail
0,6,None,2137 W 1st Ave,Completed,1984,Helen's Court Co-op Housing Association,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.1536261 49.27104183),Kitsilano,35,Clientele,Families
1,7,None,1349 E 2nd Ave,Completed,1987,Laura Jamieson Co-op Housing Association,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.07614157 49.26900321),Grandview-Woodland,42,Clientele,Families
2,8,None,1507 E 2nd Ave,Completed,1984,Westerdale Co-op Housing Association,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.07304966 49.26896961),Grandview-Woodland,10,Clientele,Families
3,18,None,1560 E 4th Ave,Completed,1989,BC Indigenous Housing Society,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.0718679 49.2666326),Grandview-Woodland,20,Clientele,Families
4,20,None,675 E 5th Ave,Completed,1985,Northern Way Co-op Housing Association,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.09065341 49.26639799),Mount Pleasant,44,Clientele,Families
...,...,...,...,...,...,...,...,...,...,...,...,...
14097,929,None,998 E 19th Ave,Approved,<NA>,NaN,NaN,POINT (-123.08438908 49.25336844),Kensington-Cedar Cottage,105,Standard,NaN
14098,934,None,1710-1730 E Pender St,Approved,<NA>,Lu'Ma Native Housing Society,NaN,POINT (-123.07002235 49.28001135),Grandview-Woodland,191,Standard,NaN
14099,948,None,545 E Cordova,Completed,2024,Lookout Housing and Health Society,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.0923813 49.28240033),Strathcona,20,Standard,NaN
14100,989,None,1119 Hornby,Completed,2017,Atira Women’s Resource Society,https://app.vancouver.ca/NonMarketHousing_NET/...,POINT (-123.12750154 49.27940628),Downtown,95,Standard,NaN


In [19]:
# drop Geom column so that altair plays nice
df = df.drop(columns=["Geom"])

units_by_clientele = alt.Chart(
    df[df["unit_category"] == "Clientele"],
    title="Total Units by Clientele"
).mark_bar().encode(
    x=alt.X("sum(unit_count):Q", title="Number of Units"),
    y=alt.Y("unit_detail:N", title="Clientele Type").sort("-x")
)

units_by_clientele.save("../img/clientele_counts.png")

units_by_clientele

alt.Chart(...)

In [20]:
family_units_by_area = alt.Chart(
    df[(df["unit_category"] == "Clientele") & (df["unit_detail"] == "Families")],
    title="Number of Family Units by Local Area"
    ).mark_bar().encode(
    y=alt.Y("Local Area").sort("-x"),
    x=alt.X("sum(unit_count):Q", title="Count of Family Units")
)

family_units_by_area.save("../img/family_by_area.png")
family_units_by_area

alt.Chart(...)